# Golden master — behavioural freeze for the Phase 2 refactor

**Step 1 of Phase 2** in `roadmap.md`. Nothing else in that phase should start
until this notebook has written a baseline.

## What it is for

Phase 2 moves the tick order, the clock, abandonment, metrics and animation
out of `SimulationBackend` and into `SimulationRunner`, then reduces
`MockBackend` to a pure adapter. That is a refactor which **shuffles
arithmetic between classes**, and its characteristic failure is not a crash —
it is a silent one-tick shift in when an arrival is deemed to happen. No unit
test in the suite would notice.

So the bar is **bit-identical output**, and this notebook is what enforces it.

## How to use it

1. **Now, before any refactoring:** run every cell. It writes
   `notebooks/golden-master.json`. Commit that file.
2. **After every step of Phase 2:** run every cell again. It compares against
   the committed baseline and reports any difference, down to the individual
   request or arrival.
3. **If a difference is intended** (it should not be, during Phase 2), set
   `ACCEPT_NEW_BASELINE = True` in the configuration cell, re-run, and commit
   the new file with an explanation of why the behaviour changed.

A green run means the refactor is behaviour-preserving. That is the only claim
this notebook makes, and it is the only one worth having here.

## Why this runs in a subprocess

`run_trial(seed=N)` seeds the global RNG, but that is **not sufficient for
reproducibility across processes**. Several policies take `min()` over a
`set[str]`, and `min` breaks ties by iteration order — which is hash
dependent. Two identical runs in different processes disagree.

That is fatal for a golden master specifically, because the whole point is to
compare a run made *today* against one made after a kernel restart, in a later
session, possibly on a different day. Without a pinned hash seed this notebook
would report differences for every scenario and be worse than useless.

`PYTHONHASHSEED` can only be set before the interpreter starts, so it cannot
be fixed from inside a running kernel — and `jupyter-run.sh` does not set it.
Rather than depend on how JupyterLab happened to be launched, the capture runs
`golden_master.py` as a **subprocess with the variable forced**.

The captured environment block records `hash_randomization` regardless, and
the comparison below refuses to run across a mismatch, so this notebook cannot
silently report an environment difference as a behavioural one.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display


def find_repo_root() -> Path:
    """
    Walk up from the working directory until the project root is found.
    :return: Path to the directory containing run.sh.
    :raises RuntimeError: If no ancestor contains run.sh.
    """
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "run.sh").exists():
            return candidate
    raise RuntimeError("could not locate the project root (no run.sh found)")


REPO_ROOT = find_repo_root()
MODULE = REPO_ROOT / "notebooks" / "golden_master.py"
BASELINE = REPO_ROOT / "notebooks" / "golden-master.json"

# Guard rail: writing a new baseline discards the reference the refactor is
# being checked against, so it must be a deliberate act.
ACCEPT_NEW_BASELINE = False

print(f"repo root: {REPO_ROOT}")
print(f"module:    {MODULE.relative_to(REPO_ROOT)}")
print(f"baseline:  {BASELINE.relative_to(REPO_ROOT)}"
      f"{'' if BASELINE.exists() else '  (does not exist yet)'}")

## 1. Capture

Roughly a minute: eleven scenarios at three seeds each. Deliberately lighter
than the experiment notebooks (120 requests, not 200) because this is a
regression check that runs after every refactor step, so turnaround matters
more than statistical power.

In [ ]:
def capture() -> dict:
    """
    Run golden_master.py in a subprocess with the hash seed pinned.

    :return: The parsed golden-master document.
    :raises RuntimeError: If the subprocess fails, with its stderr attached.
    """
    environment = {
        **os.environ,
        "PYTHONHASHSEED": "0",
        "PYTHONPATH": str(REPO_ROOT),
    }
    completed = subprocess.run(
        [sys.executable, str(MODULE)],
        cwd=REPO_ROOT,
        env=environment,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"golden_master.py failed ({completed.returncode}):\n"
            f"{completed.stderr}"
        )
    return json.loads(completed.stdout)


current = capture()
environment = current["environment"]

print(f"python              {environment['python']}")
print(f"git commit          {environment['git_commit']}")
print(f"hash randomization  {environment['hash_randomization']}")
print(f"scenarios           {len(current['scenarios'])}")
print(f"seeds               {environment['seeds']}")

## 2. What each scenario is pinning

Every scenario exists to cover a code path the refactor touches. The `why`
travels with the data, so a future session reading a diff can tell whether it
matters without reconstructing the reasoning.

In [ ]:
rows = [
    "| Scenario | fulfilled | ticks | arrivals | Covers |",
    "| --- | --- | --- | --- | --- |",
]
first_seed = str(environment["seeds"][0])
for name, block in current["scenarios"].items():
    trial = block["trials"][first_seed]
    rows.append(
        f"| `{name}` | {trial['kpi']['fulfilled_count']} "
        f"| {trial['tick_count']:,} | {len(trial['arrivals'])} "
        f"| {block['why']} |"
    )
display(Markdown("\n".join(rows)))
display(Markdown(
    "*Counts are for the first seed only, as a sanity read. Note "
    "`coarse_tick`: more arrivals than ticks, which is the multi-arrival "
    "inner loop firing — the behaviour the new `advance()` contract has to "
    "reproduce exactly.*"
))

## 3. Compare against the baseline

On the first run this writes the baseline. On every later run it compares.

Comparison is exact. Floats are compared by equality on purpose: they are
produced by the same deterministic arithmetic from the same seeds, so any
difference at all is a behavioural change, and a tolerance would only hide the
one-tick shifts this notebook exists to catch.

In [ ]:
# Fields compared per trial. Arrays are compared element-wise so a failure can
# be localised to a request or an arrival rather than merely detected.
COMPARED = (
    "kpi", "tick_count", "n_tick_history",
    "wait_times", "ride_times", "arrivals",
)


def first_difference(before: object, after: object) -> str | None:
    """
    Describe the first difference between two captured values.

    :param before: Baseline value.
    :param after: Current value.
    :return: A short human-readable description, or None when identical.
    """
    if before == after:
        return None
    if isinstance(before, dict) and isinstance(after, dict):
        for key in sorted(set(before) | set(after)):
            if before.get(key) != after.get(key):
                return f"{key}: {before.get(key)!r} -> {after.get(key)!r}"
    if isinstance(before, list) and isinstance(after, list):
        if len(before) != len(after):
            return f"length {len(before)} -> {len(after)}"
        for index, (was, now) in enumerate(zip(before, after)):
            if was != now:
                return f"index {index}: {was!r} -> {now!r}"
    return f"{before!r} -> {after!r}"


def compare(baseline: dict, current: dict) -> list[tuple[str, str, str, str]]:
    """
    Compare two golden-master documents scenario by scenario.

    :param baseline: The committed reference document.
    :param current: The document just captured.
    :return: One (scenario, seed, field, description) row per difference.
    """
    differences: list[tuple[str, str, str, str]] = []
    scenarios = sorted(
        set(baseline["scenarios"]) | set(current["scenarios"])
    )
    for name in scenarios:
        was = baseline["scenarios"].get(name)
        now = current["scenarios"].get(name)
        if was is None or now is None:
            side = "missing from baseline" if was is None else "missing now"
            differences.append((name, "-", "scenario", side))
            continue
        for seed in sorted(set(was["trials"]) | set(now["trials"])):
            before, after = was["trials"].get(seed), now["trials"].get(seed)
            if before is None or after is None:
                differences.append((name, seed, "trial", "seed set changed"))
                continue
            for field in COMPARED:
                detail = first_difference(before[field], after[field])
                if detail is not None:
                    differences.append((name, seed, field, detail))
    return differences

In [ ]:
if not BASELINE.exists() or ACCEPT_NEW_BASELINE:
    action = "Wrote" if not BASELINE.exists() else "**Replaced**"
    BASELINE.write_text(json.dumps(current, indent=1) + "\n")
    size_kb = BASELINE.stat().st_size / 1024
    display(Markdown(
        f"{action} `{BASELINE.name}` ({size_kb:.0f} KB). "
        f"Commit it — the refactor is checked against this file.\n\n"
        f"Now set `ACCEPT_NEW_BASELINE = False` so a later run cannot "
        f"overwrite it by accident."
    ))
else:
    baseline = json.loads(BASELINE.read_text())
    baseline_env = baseline["environment"]

    if baseline_env["hash_randomization"] != environment["hash_randomization"]:
        display(Markdown(
            "### Cannot compare\n\n"
            "The baseline was captured with `hash_randomization="
            f"{baseline_env['hash_randomization']}` and this run has "
            f"`{environment['hash_randomization']}`. Tie-breaking in several "
            "policies follows hash order, so the two are not comparable and "
            "any diff would be environmental rather than behavioural. "
            "Re-capture the baseline."
        ))
    else:
        differences = compare(baseline, current)
        if not differences:
            display(Markdown(
                f"### No differences\n\n"
                f"All {len(current['scenarios'])} scenarios at "
                f"{len(environment['seeds'])} seeds reproduce the baseline "
                f"captured at commit `{baseline_env['git_commit']}` exactly. "
                f"The refactor so far is behaviour-preserving."
            ))
        else:
            rows = [
                f"### {len(differences)} differences", "",
                f"Baseline commit `{baseline_env['git_commit']}` -> current "
                f"`{environment['git_commit']}`", "",
                "| Scenario | Seed | Field | First difference |",
                "| --- | --- | --- | --- |",
            ]
            for name, seed, field, detail in differences[:60]:
                rows.append(f"| `{name}` | {seed} | {field} | {detail} |")
            if len(differences) > 60:
                rows.append(
                    f"| ... | | | {len(differences) - 60} more |"
                )
            display(Markdown("\n".join(rows)))

## How to read a difference

**During Phase 2 there should be none.** Every step is a refactor; none is
supposed to change what the simulation does.

If differences appear, the `Field` column says how bad it is:

| Field | Reading |
|---|---|
| `arrivals` | The bus went somewhere else, or at a different time. The most serious: it means the movement or replanning path changed. Check the first differing index — an off-by-one-tick shift shows up as a small `sim_time` delta with identical stop IDs. |
| `wait_times` / `ride_times` | Individual passengers were served differently. Usually downstream of an `arrivals` change; if `arrivals` matches and these do not, suspect boarding or the pool. |
| `tick_count` / `n_tick_history` | The tick loop ran a different number of times — the loop condition or `record_tick` placement moved. |
| `kpi` | Aggregates only. If everything above matches and this does not, the KPI computation changed rather than the simulation. |

**Scenario tells you where.** `coarse_tick` alone failing points squarely at
the multi-arrival-per-tick contract. `abandonment` alone points at
`_expire_pending` and its position in the tick order. `dwell` alone points at
the leading-edge dwell fold. `capacity_pressure` alone points at the progress
fallback.

## What this does not cover

- **Only the mock backend.** It cannot say anything about CARLA; that is what
  the scripted backend in Phase 2 step 8 is for.
- **Only these eleven configurations.** A refactor could break something no
  scenario reaches. If a step touches a path not listed above, add a scenario
  to `golden_master.py` *before* making the change, and re-baseline.
- **Not a measurement.** 120 requests at three seeds is a fingerprint, not an
  experiment. Do not read the KPIs here as results.